# Visualize pipeline for one frame

This notebook runs the SurgeSlam pipeline for a single frame (configurable) and shows inputs/outputs at each step to make debugging easier.

Sections:
- Imports and setup
- Helper functions
- Notebook configuration (pick run folder + frame index)
- Step-by-step pipeline execution (input load -> depth pred -> deformation -> transform -> render)
- Visualizations and parameter inspections
- Save outputs

Run the cells in order. If an import fails, run the notebook from the repository root with the project's Python environment active.

In [1]:
# Cell 1: Imports and environment (robust repo root detection)
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io
import torch
import torchvision

# Robustly find the repository root by walking up until we find a 'scripts' package or reach filesystem root
def find_repo_root(start_path: str = None, marker_names=('scripts', '.git', 'setup.py')) -> str:
    p = os.path.abspath(start_path or os.getcwd())
    last = None
    while p != last:
        # if any marker exists directly under p, consider this the repo root
        for m in marker_names:
            candidate = os.path.join(p, m)
            if os.path.exists(candidate):
                return p
        last = p
        p = os.path.dirname(p)
    # fallback to cwd if nothing found
    return os.path.abspath(start_path or os.getcwd())

ROOT = find_repo_root()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print('Repo root set to', ROOT)

# Project imports (best-effort) - these should resolve when ROOT is the repo root
try:
    from scripts.main_SurgeSplat import deform_gaussians, transform_to_frame, transformed_params2depthplussilhouette, transformed_params2rendervar
    from utils.common_utils import params2cpu
    from viz_scripts.final_recon import Renderer
    print('Project imports OK')
except Exception as e:
    print('Project imports failed:', e)
    print('sys.path currently:', sys.path[:6])

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)


Repo root set to /home/nasr/SurgeSplam
System Paths:
/home/nasr/SurgeSplam
/home/nasr/SurgeSplam
/home/nasr/miniconda/envs/endogslam/lib/python310.zip
/home/nasr/miniconda/envs/endogslam/lib/python3.10
/home/nasr/miniconda/envs/endogslam/lib/python3.10/lib-dynload

/home/nasr/miniconda/envs/endogslam/lib/python3.10/site-packages


xFormers not available
xFormers not available
xFormers not available


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Project imports OK
Project imports OK


In [2]:
# Cell 2: Helper functions
from typing import Optional

import numpy as _np
import torch as _torch


def convert_param_to_tensor(v, device='cpu', require_grad=False):
    """Safely convert a numpy parameter to a torch tensor on `device`.
    - If v is object-dtype array, try to convert elements to numeric arrays, otherwise return Python list (leave as non-tensor).
    - If v is list/tuple, try to convert to numeric array first.
    - Returns either a torch.Tensor (with requires_grad set) or the original Python object/list.
    """
    # numpy arrays
    if isinstance(v, _np.ndarray):
        # handle object-dtype arrays (e.g., arrays of arrays or lists)
        if v.dtype == _np.object_:
            try:
                # try to coerce each element to ndarray and stack
                arrs = _np.array([_np.asarray(x) for x in v])
                # If arrs is numeric, convert to tensor
                if arrs.dtype.kind in ('f', 'i', 'u', 'c'):
                    return _torch.tensor(arrs, dtype=_torch.float32, device=device).requires_grad_(require_grad)
                else:
                    # fallback to python list
                    return [(_np.asarray(x).tolist() if not isinstance(x, (list, tuple)) and hasattr(x, 'tolist') else x) for x in v.tolist()]
            except Exception:
                return v.tolist()
        else:
            return _torch.tensor(v, dtype=_torch.float32, device=device).requires_grad_(require_grad)
    # lists / tuples
    if isinstance(v, (list, tuple)):
        try:
            arr = _np.asarray(v)
            if arr.dtype == _np.object_:
                return v
            return _torch.tensor(arr, dtype=_torch.float32, device=device).requires_grad_(require_grad)
        except Exception:
            return v
    # other types: return as-is
    return v


def load_image(path: str) -> _np.ndarray:
    if path is None:
        raise FileNotFoundError('No path provided')
    img = Image.open(path).convert('RGB')
    return _np.array(img)


def load_depth(path: str) -> _np.ndarray:
    if path.endswith('.npy'):
        return _np.load(path)
    if path.endswith('.npz'):
        with _np.load(path) as d:
            return d[list(d.keys())[0]]
    img = Image.open(path)
    arr = _np.array(img)
    if arr.ndim == 3:
        return arr[...,0].astype(_np.float32)
    return arr.astype(_np.float32)


def normalize_depth(d: _np.ndarray) -> _np.ndarray:
    d = _np.array(d, dtype=_np.float32)
    valid = d[_np.isfinite(d)]
    if valid.size == 0:
        return d
    dmin = float(valid.min())
    dmax = float(valid.max())
    if dmax == dmin:
        return _np.zeros_like(d)
    return (d - dmin) / (dmax - dmin)


def depth_to_colormap(d_norm: _np.ndarray, cmap='viridis') -> _np.ndarray:
    cmapf = plt.get_cmap(cmap)
    rgba = cmapf(d_norm)
    return (rgba[..., :3] * 255).astype(_np.uint8)

print('Helpers ready')

Helpers ready


In [3]:
# Cell 3: Notebook configuration - pick experiment folder and frame
from ipywidgets import widgets

exp_default = os.path.join(ROOT, 'experiments', 'C3VDv2_base', 'c1_transverse1_t4_v4')
exp_folder = widgets.Text(description='exp_folder', value=exp_default)
frame_idx = widgets.IntText(description='frame_idx', value=1)

exp_folder, frame_idx


def resolve_paths(exp_folder: str, frame: int):
    paths = {}
    params0 = os.path.join(exp_folder, 'params0.npz')
    params1 = os.path.join(exp_folder, 'params.npz')
    bad = os.path.join(exp_folder, 'bad_checkpoints', 'params.bad.npz')
    paths['params_path'] = None
    for p in (params0, bad, params1):
        if os.path.exists(p):
            paths['params_path'] = p
            break
    # guess input image from eval folder
    eval_dir = os.path.join(exp_folder, 'eval')
    paths['input_image'] = None
    if os.path.isdir(eval_dir):
        imgs = [os.path.join(eval_dir, f) for f in os.listdir(eval_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        imgs.sort()
        if len(imgs) > frame:
            paths['input_image'] = imgs[frame]
    return paths

print('Config ready; update widgets above as needed and run the next cell')

Config ready; update widgets above as needed and run the next cell


In [4]:
# Cell 5: Recreate model from config + run one training iteration and show before/after renders
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# Choose which params to treat as trainable for this smoke iteration
trainable_keys = ['means3D','unnorm_rotations','log_scales','logit_opacities','rgb_colors']

# Build torch params dict; keep non-trainable tensors detached
torch_params_train = {}
for k, v in params.items():
    t = convert_param_to_tensor(v, device=device.type, require_grad=(k in trainable_keys))
    torch_params_train[k] = t

# If w2c (world-to-camera) missing, try loading from eval/gt_train_w2c.txt or eval/est_w2c.txt
if ('w2c' not in torch_params_train) or (torch_params_train.get('w2c') is None):
    eval_dir = os.path.join(exp_folder.value, 'eval')
    candidates = [os.path.join(eval_dir, 'gt_train_w2c.txt'), os.path.join(eval_dir, 'est_w2c.txt')]
    loaded_w2c = False
    for p in candidates:
        if os.path.exists(p):
            try:
                with open(p, 'r') as fh:
                    lines = [ln.strip() for ln in fh.readlines() if ln.strip()]
                if len(lines) > frame_idx.value:
                    nums = [float(x) for x in lines[frame_idx.value].split(',')]
                    if len(nums) >= 16:
                        arr = np.array(nums, dtype=np.float32).reshape(4,4)
                        torch_params_train['w2c'] = torch.tensor(arr, device=device)
                        print(f"Loaded w2c for frame {frame_idx.value} from {p}")
                        loaded_w2c = True
                        break
            except Exception as e:
                print('Failed to parse w2c file', p, e)
    if not loaded_w2c:
        print('Warning: w2c not found in params or eval files; rendering may fail')

# Build optimizer over the selected trainable tensors
opt_param_list = [p for p in torch_params_train.values() if isinstance(p, torch.Tensor) and p.requires_grad]
if len(opt_param_list) == 0:
    raise RuntimeError('No trainable parameters found. Add keys to trainable_keys or make sure params contain those keys.')
optimizer = torch.optim.Adam(opt_param_list, lr=1e-3)
print('Optimizer created with', len(opt_param_list), 'param tensors')

# Helper to safely render given a params dict
def render_from_params(tp):
    # Expect tp has tensors on correct device and required keys
    try:
        local_means, local_rots, local_scales, local_opacities, local_colors = deform_gaussians(tp, frame_idx.value, deform_grad=True, deformation_type='gaussian')
        transformed_pts = transform_to_frame(local_means, tp, frame_idx.value, gaussians_grad=True, camera_grad=False)
        # Build depth/silhouette rendervar (many pipelines use same rendervar for rgb too)
        rendervar_depth = transformed_params2depthplussilhouette(tp, tp.get('w2c', None), transformed_pts, local_rots, local_scales, local_opacities)
        # Call renderer. Ensure we don't pass None as raster_settings (some Renderer implementations
        # expect a raster_settings object with attributes like 'bg'). If cam is missing, try to
        # create a default Renderer() without explicit raster_settings so it uses defaults.
        cam_setting = tp.get('cam', None)
        if cam_setting is None:
            try:
                renderer_fn = Renderer()
            except Exception as e:
                print('Failed to create default Renderer()', e)
                # Fallback: attempt to pass cam_setting (None) to force an informative error
                renderer_fn = Renderer(raster_settings=cam_setting)
        else:
            renderer_fn = Renderer(raster_settings=cam_setting)
        rendered_depth, rendered_im, extra = renderer_fn(**rendervar_depth)
        return rendered_depth, rendered_im, extra
    except Exception as e:
        print('Rendering error:', e)
        raise

# Prepare input tensor if available
input_t = None
if input_img is not None:
    # Normalize to [0,1] and permute to C,H,W
    input_t = torch.tensor(np.array(input_img)/255.0, dtype=torch.float32, device=device).permute(2,0,1)
    print('Input tensor prepared:', input_t.shape)
else:
    print('No input image; will run optimization without photometric loss (may use silhouette if available)')

# Render BEFORE update
print('\nRendering BEFORE update...')
rendered_depth_b, rendered_im_b, extra_b = render_from_params(torch_params_train)
# rendered_im_b likely in [C,H,W] (float)
if rendered_im_b.dim() == 4:
    # remove batch dim
    rendered_im_b = rendered_im_b[0]
if rendered_depth_b.dim() == 3 and rendered_depth_b.shape[0] == 1:
    rendered_depth_b = rendered_depth_b[0]

# Compute loss (if input available)
loss = None
if input_t is not None:
    # Resize or crop input to match rendered size if necessary
    r = rendered_im_b.detach()
    if input_t.shape[1:] != r.shape[1:]:
        input_resized = F.interpolate(input_t.unsqueeze(0), size=r.shape[1:], mode='bilinear', align_corners=False)[0]
    else:
        input_resized = input_t
    # If renderer returns values not clamped, clamp to 0..1
    pred = torch.clamp(rendered_im_b, 0.0, 1.0)
    loss = F.mse_loss(pred, input_resized)
    print('Photometric loss BEFORE:', float(loss.detach().cpu().item()))
else:
    # Try silhouette if available to create a proxy loss (encourage silhouette match to 0.5)
    if extra_b is not None and isinstance(extra_b, dict) and 'silhouette' in extra_b:
        sil = extra_b['silhouette']
        if sil.dim() == 4:
            sil = sil[0]
        target = torch.zeros_like(sil)
        loss = F.mse_loss(sil, target)
        print('Silhouette proxy loss BEFORE:', float(loss.detach().cpu().item()))
    else:
        print('No suitable loss (no input image or silhouette); skipping optimization step')

# Run one optimization step if we have a loss
if loss is not None:
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(opt_param_list, 1.0)
    optimizer.step()
    print('Optimizer step completed')

# Render AFTER update
print('\nRendering AFTER update...')
rendered_depth_a, rendered_im_a, extra_a = render_from_params(torch_params_train)
if rendered_im_a.dim() == 4:
    rendered_im_a = rendered_im_a[0]
if rendered_depth_a.dim() == 3 and rendered_depth_a.shape[0] == 1:
    rendered_depth_a = rendered_depth_a[0]

# Convert to numpy for display
def to_uint8_img(t):
    t = t.detach().cpu()
    if t.dim() == 3:
        t = t.permute(1,2,0)
    arr = (torch.clamp(t,0.0,1.0).numpy()*255).astype(np.uint8)
    return arr

rgb_b = to_uint8_img(rendered_im_b)
rgb_a = to_uint8_img(rendered_im_a)
depth_b = rendered_depth_b.detach().cpu().numpy() if isinstance(rendered_depth_b, torch.Tensor) else np.array(rendered_depth_b)
depth_a = rendered_depth_a.detach().cpu().numpy() if isinstance(rendered_depth_a, torch.Tensor) else np.array(rendered_depth_a)

# Display side-by-side
import matplotlib.pyplot as plt
fig, axs = plt.subplots(2,3, figsize=(18,12))
# Row 0: BEFORE
if input_img is not None:
    axs[0,0].imshow(input_img)
    axs[0,0].set_title('Input RGB')
else:
    axs[0,0].text(0.5,0.5,'No input',ha='center')
axs[0,1].imshow(rgb_b)
axs[0,1].set_title('Rendered RGB BEFORE')
axs[0,2].imshow(depth_to_colormap(normalize_depth(depth_b)))
axs[0,2].set_title('Rendered Depth BEFORE')
# Row 1: AFTER
axs[1,0].axis('off')
axs[1,1].imshow(rgb_a)
axs[1,1].set_title('Rendered RGB AFTER')
axs[1,2].imshow(depth_to_colormap(normalize_depth(depth_a)))
axs[1,2].set_title('Rendered Depth AFTER')
for ax in axs.ravel():
    ax.axis('off')
plt.show()

# Save outputs
out_dir = os.path.join(exp_folder.value, 'debug_oneframe')
os.makedirs(out_dir, exist_ok=True)
from PIL import Image as PILImage
PILImage.fromarray(rgb_b).save(os.path.join(out_dir, f'rendered_before_frame{frame_idx.value}.png'))
PILImage.fromarray(rgb_a).save(os.path.join(out_dir, f'rendered_after_frame{frame_idx.value}.png'))
np.save(os.path.join(out_dir, f'rendered_depth_before_frame{frame_idx.value}.npy'), depth_b)
np.save(os.path.join(out_dir, f'rendered_depth_after_frame{frame_idx.value}.npy'), depth_a)
print('Saved before/after outputs to', out_dir)


Using device: cuda


NameError: name 'params' is not defined

In [7]:
# Cell 5: Recreate model from config + run one training iteration and show before/after renders
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# Choose which params to treat as trainable for this smoke iteration
trainable_keys = ['means3D','unnorm_rotations','log_scales','logit_opacities','rgb_colors']

# Build torch params dict; keep non-trainable tensors detached
torch_params_train = {}
for k, v in params.items():
    t = convert_param_to_tensor(v, device=device.type, require_grad=(k in trainable_keys))
    torch_params_train[k] = t

# If w2c (world-to-camera) missing, try loading from eval/gt_train_w2c.txt or eval/est_w2c.txt
if ('w2c' not in torch_params_train) or (torch_params_train.get('w2c') is None):
    eval_dir = os.path.join(exp_folder.value, 'eval')
    candidates = [os.path.join(eval_dir, 'gt_train_w2c.txt'), os.path.join(eval_dir, 'est_w2c.txt')]
    loaded_w2c = False
    for p in candidates:
        if os.path.exists(p):
            try:
                with open(p, 'r') as fh:
                    lines = [ln.strip() for ln in fh.readlines() if ln.strip()]
                if len(lines) > frame_idx.value:
                    nums = [float(x) for x in lines[frame_idx.value].split(',')]
                    if len(nums) >= 16:
                        arr = np.array(nums, dtype=np.float32).reshape(4,4)
                        torch_params_train['w2c'] = torch.tensor(arr, device=device)
                        print(f"Loaded w2c for frame {frame_idx.value} from {p}")
                        loaded_w2c = True
                        break
            except Exception as e:
                print('Failed to parse w2c file', p, e)
    if not loaded_w2c:
        print('Warning: w2c not found in params or eval files; rendering may fail')

# Build optimizer over the selected trainable tensors
opt_param_list = [p for p in torch_params_train.values() if isinstance(p, torch.Tensor) and p.requires_grad]
if len(opt_param_list) == 0:
    raise RuntimeError('No trainable parameters found. Add keys to trainable_keys or make sure params contain those keys.')
optimizer = torch.optim.Adam(opt_param_list, lr=1e-3)
print('Optimizer created with', len(opt_param_list), 'param tensors')

# Helper to safely render given a params dict
def render_from_params(tp):
    # Expect tp has tensors on correct device and required keys
    try:
        local_means, local_rots, local_scales, local_opacities, local_colors = deform_gaussians(tp, frame_idx.value, deform_grad=True, deformation_type='gaussian')
        transformed_pts = transform_to_frame(local_means, tp, frame_idx.value, gaussians_grad=True, camera_grad=False)
        # Build depth/silhouette rendervar (many pipelines use same rendervar for rgb too)
        rendervar_depth = transformed_params2depthplussilhouette(tp, tp.get('w2c', None), transformed_pts, local_rots, local_scales, local_opacities)
        # Call renderer
        rendered_depth, rendered_im, extra = Renderer(raster_settings=tp.get('cam', None))(**rendervar_depth)
        return rendered_depth, rendered_im, extra
    except Exception as e:
        print('Rendering error:', e)
        raise

# Prepare input tensor if available
input_t = None
if input_img is not None:
    # Normalize to [0,1] and permute to C,H,W
    input_t = torch.tensor(np.array(input_img)/255.0, dtype=torch.float32, device=device).permute(2,0,1)
    print('Input tensor prepared:', input_t.shape)
else:
    print('No input image; will run optimization without photometric loss (may use silhouette if available)')

# Render BEFORE update
print('\nRendering BEFORE update...')
rendered_depth_b, rendered_im_b, extra_b = render_from_params(torch_params_train)
# rendered_im_b likely in [C,H,W] (float)
if rendered_im_b.dim() == 4:
    # remove batch dim
    rendered_im_b = rendered_im_b[0]
if rendered_depth_b.dim() == 3 and rendered_depth_b.shape[0] == 1:
    rendered_depth_b = rendered_depth_b[0]

# Compute loss (if input available)
loss = None
if input_t is not None:
    # Resize or crop input to match rendered size if necessary
    r = rendered_im_b.detach()
    if input_t.shape[1:] != r.shape[1:]:
        input_resized = F.interpolate(input_t.unsqueeze(0), size=r.shape[1:], mode='bilinear', align_corners=False)[0]
    else:
        input_resized = input_t
    # If renderer returns values not clamped, clamp to 0..1
    pred = torch.clamp(rendered_im_b, 0.0, 1.0)
    loss = F.mse_loss(pred, input_resized)
    print('Photometric loss BEFORE:', float(loss.detach().cpu().item()))
else:
    # Try silhouette if available to create a proxy loss (encourage silhouette match to 0.5)
    if extra_b is not None and isinstance(extra_b, dict) and 'silhouette' in extra_b:
        sil = extra_b['silhouette']
        if sil.dim() == 4:
            sil = sil[0]
        target = torch.zeros_like(sil)
        loss = F.mse_loss(sil, target)
        print('Silhouette proxy loss BEFORE:', float(loss.detach().cpu().item()))
    else:
        print('No suitable loss (no input image or silhouette); skipping optimization step')

# Run one optimization step if we have a loss
if loss is not None:
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(opt_param_list, 1.0)
    optimizer.step()
    print('Optimizer step completed')

# Render AFTER update
print('\nRendering AFTER update...')
rendered_depth_a, rendered_im_a, extra_a = render_from_params(torch_params_train)
if rendered_im_a.dim() == 4:
    rendered_im_a = rendered_im_a[0]
if rendered_depth_a.dim() == 3 and rendered_depth_a.shape[0] == 1:
    rendered_depth_a = rendered_depth_a[0]

# Convert to numpy for display
def to_uint8_img(t):
    t = t.detach().cpu()
    if t.dim() == 3:
        t = t.permute(1,2,0)
    arr = (torch.clamp(t,0.0,1.0).numpy()*255).astype(np.uint8)
    return arr

rgb_b = to_uint8_img(rendered_im_b)
rgb_a = to_uint8_img(rendered_im_a)
depth_b = rendered_depth_b.detach().cpu().numpy() if isinstance(rendered_depth_b, torch.Tensor) else np.array(rendered_depth_b)
depth_a = rendered_depth_a.detach().cpu().numpy() if isinstance(rendered_depth_a, torch.Tensor) else np.array(rendered_depth_a)

# Display side-by-side
import matplotlib.pyplot as plt
fig, axs = plt.subplots(2,3, figsize=(18,12))
# Row 0: BEFORE
if input_img is not None:
    axs[0,0].imshow(input_img)
    axs[0,0].set_title('Input RGB')
else:
    axs[0,0].text(0.5,0.5,'No input',ha='center')
axs[0,1].imshow(rgb_b)
axs[0,1].set_title('Rendered RGB BEFORE')
axs[0,2].imshow(depth_to_colormap(normalize_depth(depth_b)))
axs[0,2].set_title('Rendered Depth BEFORE')
# Row 1: AFTER
axs[1,0].axis('off')
axs[1,1].imshow(rgb_a)
axs[1,1].set_title('Rendered RGB AFTER')
axs[1,2].imshow(depth_to_colormap(normalize_depth(depth_a)))
axs[1,2].set_title('Rendered Depth AFTER')
for ax in axs.ravel():
    ax.axis('off')
plt.show()

# Save outputs
out_dir = os.path.join(exp_folder.value, 'debug_oneframe')
os.makedirs(out_dir, exist_ok=True)
from PIL import Image as PILImage
PILImage.fromarray(rgb_b).save(os.path.join(out_dir, f'rendered_before_frame{frame_idx.value}.png'))
PILImage.fromarray(rgb_a).save(os.path.join(out_dir, f'rendered_after_frame{frame_idx.value}.png'))
np.save(os.path.join(out_dir, f'rendered_depth_before_frame{frame_idx.value}.npy'), depth_b)
np.save(os.path.join(out_dir, f'rendered_depth_after_frame{frame_idx.value}.npy'), depth_a)
print('Saved before/after outputs to', out_dir)

Using device: cuda
Loaded w2c for frame 1 from /home/nasr/SurgeSplam/experiments/C3VDv2_base/c1_transverse1_t4_v4/eval/gt_train_w2c.txt
Optimizer created with 5 param tensors
No input image; will run optimization without photometric loss (may use silhouette if available)

Rendering BEFORE update...
Rendering error: 'NoneType' object has no attribute 'bg'
Rendering error: 'NoneType' object has no attribute 'bg'


AttributeError: 'NoneType' object has no attribute 'bg'

In [ ]:
# Cell 5: Step-by-step pipeline run for one frame
# Steps: (1) optional depth prediction (if model exists) (2) deform_gaussians -> (3) transform_to_frame -> (4) build rendervar -> (5) Renderer

if params is None:
    raise RuntimeError('Params not loaded. Run the previous cell and ensure params_path is set.')

# Convert numpy params to torch tensors if needed
torch_params = {}
for k, v in params.items():
    if isinstance(v, np.ndarray):
        try:
            torch_params[k] = torch.tensor(v).cuda().float().requires_grad_(False)
        except Exception:
            torch_params[k] = torch.tensor(v).float().requires_grad_(False)
    else:
        torch_params[k] = v

# 1) Optional: show some raw params that often cause problems
inspect_keys = ['means3D','unnorm_rotations','log_scales','logit_opacities','cv_vel_xyz','fourier_xyz_cos']
print('\nParameter check:')
for k in inspect_keys:
    if k in params:
        v = params[k]
        print(f"{k}: shape={getattr(v,'shape',None)} dtype={getattr(v,'dtype',None)} finite={np.isfinite(v).all() if isinstance(v,np.ndarray) else 'unknown'}")

# 2) Deformation
print('\nRunning deform_gaussians...')
local_means, local_rots, local_scales, local_opacities, local_colors = deform_gaussians(torch_params, frame_idx.value, deform_grad=False, deformation_type='gaussian')
print('Deform outputs shapes:', local_means.shape, local_rots.shape, local_scales.shape)

# Show a quick scatter of a downsampled subset of means
mm = local_means.detach().cpu().numpy()
if mm.ndim == 3:
    # shape assumed (G, 3, T) or similar - try to reduce
    try:
        pts = mm[:, :, frame_idx.value]
    except Exception:
        pts = mm.reshape(-1, 3)[:10000]
else:
    pts = mm[:10000]

fig = plt.figure(figsize=(6,6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(pts[:2000,0], pts[:2000,1], pts[:2000,2], s=1)
ax.set_title('Downsampled local_means (3D preview)')
plt.show()

# 3) Transform to frame
print('\nRunning transform_to_frame...')
transformed_pts = transform_to_frame(local_means, torch_params, frame_idx.value, gaussians_grad=False, camera_grad=False)
print('transformed_pts shape:', transformed_pts.shape)

# 4) Build rendervar for depth+silhouette
print('\nBuilding depth+silhouette rendervar...')
rendervar_depth = transformed_params2depthplussilhouette(torch_params, torch_params.get('w2c', None), transformed_pts, local_rots, local_scales, local_opacities)
print('rendervar keys:', list(rendervar_depth.keys()))

# 5) Render
print('\nRendering...')
# Ensure raster_settings passed to Renderer is not None to avoid AttributeError in C++/CUDA rasterizer
cam_setting = torch_params.get('cam', None)
if cam_setting is None:
    try:
        # Try constructing Renderer with defaults if cam is missing
        renderer = Renderer()
    except Exception as e:
        print('Failed to create default Renderer()', e)
        # As a fallback, attempt to pass cam_setting anyway (will likely raise); let the error surface
        renderer = Renderer(raster_settings=cam_setting)
else:
    renderer = Renderer(raster_settings=cam_setting)
rendered_depth, rendered_im, extra = renderer(**rendervar_depth)
print('Rendered shapes:', rendered_im.shape, rendered_depth.shape)

# Convert to numpy for display
rendered_im_np = (rendered_im.permute(1,2,0).cpu().detach().numpy()*255).astype(np.uint8)
rendered_depth_np = rendered_depth[0,:,:].cpu().detach().numpy()

# Display results side-by-side
fig, axs = plt.subplots(1,3, figsize=(18,6))
if input_img is not None:
    axs[0].imshow(input_img)
    axs[0].set_title('Input RGB')
    axs[0].axis('off')
else:
    axs[0].text(0.5, 0.5, 'No input image', ha='center')

axs[1].imshow(rendered_im_np)
axs[1].set_title('Rendered RGB')
axs[1].axis('off')

dn = normalize_depth(rendered_depth_np)
axs[2].imshow(depth_to_colormap(dn))
axs[2].set_title('Rendered Depth (colormap)')
axs[2].axis('off')

plt.show()

print('\nRendered depth stats: min, max, mean, nan_count, inf_count')
print(np.nanmin(rendered_depth_np), np.nanmax(rendered_depth_np), np.nanmean(rendered_depth_np), np.isnan(rendered_depth_np).sum(), np.isinf(rendered_depth_np).sum())

# Save some outputs to experiment folder for inspection
out_dir = os.path.join(exp_folder.value, 'debug_oneframe')
os.makedirs(out_dir, exist_ok=True)
from PIL import Image as PILImage
PILImage.fromarray(rendered_im_np).save(os.path.join(out_dir, f'rendered_frame{frame_idx.value}.png'))
np.save(os.path.join(out_dir, f'rendered_depth_frame{frame_idx.value}.npy'), rendered_depth_np)
print('Saved rendered images to', out_dir)


TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint8, and bool.